<a href="https://colab.research.google.com/github/KeertanaOfficial/AgenticAIProjects/blob/main/Notebooks/Autonomous_IT_Support_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The Autonomous IT Support Agent**

**Objective:** You are building an AI agent that acts as the "first responder" for server incidents. It must:

1. **Investigate:** Check server health and logs when a user reports an issue.
2. **Act:** If CPU is critical (>90%), it should **Restart** the service.
3. **Escalate:** If the issue is complex or logs show a major problem it should **Escalate** to a human.

In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [2]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

# Initialize Client

==========================================
## PART 1: DEFINE THE TOOLS (BUSINESS LOGIC)
==========================================

In [3]:
# --- Tool 1: Check Health ---
def get_server_health(server_id: str) -> str:
    """Returns CPU and Memory usage for a given server."""
    print(f"-> TOOL: Checking health for {server_id}...")

    metrics = {
        # Scenario 1: High CPU (Needs Restart)
        "payment-server-01": {"cpu": "98%", "memory": "40%", "status": "Warning"},

        # Scenario 2: Healthy (No Action Needed)
        "db-node-02": {"cpu": "12%", "memory": "60%", "status": "Healthy"},

        # Scenario 3: High Memory Leak (Needs Restart or Escalation)
        "auth-service-03": {"cpu": "45%", "memory": "95%", "status": "Critical"},

        # Scenario 4: Network/Dependency Failure (Needs Escalation)
        "search-index-09": {"cpu": "10%", "memory": "15%", "status": "Error"},

        # Scenario 5: Completely Normal
        "frontend-node-04": {"cpu": "25%", "memory": "30%", "status": "Healthy"},
    }

    result = metrics.get(server_id, {"error": "Server not found. Check the ID."})
    return json.dumps(result)


In [4]:
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the last N lines of logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")

    # Different logs for different servers to trigger different agent behaviors
    log_database = {
        "payment-server-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool exhaustion",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-node-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 12ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-service-03": [
            "[INFO] Token validated user_882",
            "[WARN] Garbage collection taking too long (>5s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashing due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-index-09": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4432",
            "[CRITICAL] Dependency Unreachable: Search Engine is down",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-04": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }

    # Default logs if server not found in specific list
    default_logs = ["[INFO] System stable", "[INFO] Heartbeat signal received"]

    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

---
### ----- Implement code below -----
---


In [20]:
# --- TASK 1: Implement the Restart Tool ---
def restart_service(server_id: str) -> str:
    """
    ### TODO: Implement this function.
    1. Print a message saying "-> TOOL: Restarting service..."
    2. Return a JSON string confirming the restart was successful.
       Example return: '{"status": "success", "message": "Server restarted successfully"}'
    """
    result = {
        "server_id": server_id,
        "status": "success",
        "message": "Server restarted successfully"
    }
    print(f"-> TOOL: Restarting service for {server_id}...")
    return json.dumps(result)

In [24]:
# --- TASK 2: Implement the Escalation Tool ---
def escalate_to_engineer(summary: str) -> str:
    """
    ### TODO: Implement this function.
    1. Print a message saying "-> TOOL: Escalating to human..."
    2. Return a JSON string confirming the ticket was created.
    """
    result = {
        "status": "Escalated",
        "ticket_id": "12345",
        "assigned_to": "Engineer"
    }
    print(f"-> TOOL: Escalating to human...",{summary})
    return json.dumps(result)


In [22]:
# Map functions for the agent execution loop
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

==========================================
## PART 2: DEFINE THE AGENT SCHEMA
==========================================


In [23]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks the current CPU and memory usage of a specific server.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Retrieves the most recent log entries from a server to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "This the is service to restart the server",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail or the error is unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                     "summary": {"type": "string", "description": "A brief summary of the issue."}
                },
                "required": ["summary"]
            }
        }
    }
]


 ==========================================
## PART 3: THE AGENT EXECUTION LOOP
 ==========================================


In [27]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")

    messages = [
        {"role": "system", "content": "You are a Level 1 IT Responder. Investigate server issues. "
                                      "If CPU or Memory is > 90%, restart the service. If logs show critical dependency errors (like connection refused) that a restart won't fix, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # Retrieve the actual python function based on name
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)

                if function_to_call:
                    # Execute the function
                    tool_output = function_to_call(**func_args)
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": tool_output
                    })
        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content} and Function name is : {response_msg}")
            break

 ==========================================
## PART 4: TEST SCENARIOS
 ==========================================


In [ ]:
from openai import OpenAI
import os

#client = OpenAI(api_key=os.getenv("OPENAI_APIKEY"))
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
models = client.models.list()

for model in models.data:
  print(model.id)

gpt-4o


In [28]:
# Scenario A: Should trigger a restart (CPU is 98%)
run_it_agent("The payment-server-01 is extremely slow and timing out.")


--- New Incident: The payment-server-01 is extremely slow and timing out. ---

[AI Thinking...]
-> TOOL: Checking health for payment-server-01...
-> TOOL: Fetching last 10 log lines for payment-server-01...

[AI Thinking...]
-> TOOL: Restarting service for payment-server-01...

[AI Thinking...]

[FINAL RESPONSE]: The payment-server-01 was experiencing high CPU usage at 98%, which was causing performance issues and timeouts. I have restarted the server service, and the restart was successful. Please monitor the server to ensure it's running smoothly now. and Function name is : ChatCompletionMessage(content="The payment-server-01 was experiencing high CPU usage at 98%, which was causing performance issues and timeouts. I have restarted the server service, and the restart was successful. Please monitor the server to ensure it's running smoothly now.", refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [ ]:
# Scenario B: Should trigger an escalation (DB is healthy but logs might be weird)
run_it_agent("Something is wrong with db-node-02")


--- New Incident: Something is wrong with db-node-02 ---

[AI Thinking...]
-> TOOL: Checking health for db-node-02...
-> TOOL: Fetching last 10 log lines for db-node-02...

[AI Thinking...]

[FINAL RESPONSE]: The server health of "db-node-02" looks good as the CPU usage is at 12% and memory usage is at 60%, both of which are within normal limits. The recent logs also show normal activity with backups running successfully, queries executing promptly, and ongoing replication synchronization.

Since there are no apparent issues with the CPU, memory, or logs, it might be worth checking any external systems or applications interacting with "db-node-02" or see if there is a specific issue that hasn't been captured in the recent logs. If there's a particular symptom or problem you're facing, let me know so I can further assist you.


In [ ]:
# Scenario C: The High Memory Case (auth-service-03)
# Agent should see Memory 95% + OutOfMemoryError logs -> Restart
run_it_agent("Users are reporting login failures on auth-service-03.")

print("\n" + "="*50 + "\n")


--- New Incident: Users are reporting login failures on auth-service-03. ---

[AI Thinking...]
-> TOOL: Checking health for auth-service-03...
-> TOOL: Fetching last 10 log lines for auth-service-03...

[AI Thinking...]
-> TOOL: Restarting service for auth-service-03...

[AI Thinking...]

[FINAL RESPONSE]: The auth-service-03 server was experiencing a memory issue, with usage at 95%, along with a critical error indicating a memory leak. I have successfully restarted the service to address the problem. Please monitor to ensure that the login functionality is restored and let me know if there are any further issues.




In [ ]:
# Scenario D: The Dependency Failure (search-index-09)
# Agent should see healthy CPU but "Connection Refused" logs -> Escalate
run_it_agent("Search isn't working. Can you check search-index-09?")

print("\n" + "="*50 + "\n")


--- New Incident: Search isn't working. Can you check search-index-09? ---

[AI Thinking...]
-> TOOL: Checking health for search-index-09...
-> TOOL: Fetching last 10 log lines for search-index-09...

[AI Thinking...]
-> TOOL: Escalating to human...

[AI Thinking...]

[FINAL RESPONSE]: The issue with the search functionality on server **search-index-09** is due to a critical dependency error: 'Connection refused' to `elastic-cluster-main:9200`. I have escalated this to an engineer since it's not a resource-related problem and needs further expertise. A ticket has been created successfully for further investigation.




In [ ]:
# Scenario E: The Healthy Server (frontend-node-04)
# Agent should see normal stats and 200 OK logs -> Do nothing / Report healthy
run_it_agent("Check frontend-node-04 just to be safe.")


--- New Incident: Check frontend-node-04 just to be safe. ---

[AI Thinking...]
-> TOOL: Checking health for frontend-node-04...
-> TOOL: Fetching last 10 log lines for frontend-node-04...

[AI Thinking...]

[FINAL RESPONSE]: The server `frontend-node-04` is operating normally. The CPU is at 25% and memory usage is at 30%, indicating no significant resource issues. The recent logs reflect normal operations with successful responses, and a health check has passed. There are no actions required at this time.


In [16]:
run_it_agent("Service restart needed for payment-server-01")


--- New Incident: Service restart needed for payment-server-01 ---

[AI Thinking...]
-> TOOL: Checking health for payment-server-01...
-> TOOL: Fetching last 20 log lines for payment-server-01...

[AI Thinking...]
-> TOOL: Restarting service for payment-server-01...

[AI Thinking...]

[FINAL RESPONSE]: The high CPU usage on `payment-server-01` was causing issues with processing. I have restarted the service, and it was completed successfully.  If you encounter further issues, please let me know.
